# Phase 1: Model Tuning & Optimization## TSMC (2330.TW) LSTM + Attention — Hyperparameter Experiments| Experiment | Variable | Values Tested ||---|---|---|| Baseline | Default settings | look_back=100, LSTM[128,64], LR=0.001, Batch=32, Epochs=50, No Dropout || Exp 1 | Sequence Length | 60, 90, 120 || Exp 2 | LSTM Architecture | [64,32], [256,128], [128,64,32], [256,128,64] || Exp 3 | Training Parameters | LR={0.0005, 0.005}, Batch={16, 64}, Epochs=100 || Exp 4 | Dropout Rate | 0.1, 0.2, 0.3 || Feature Selection | Multi-feature (PyTorch) | 7 technical indicators |

In [ ]:
# ============================================================
# Package Installation
# ============================================================
import subprocess, sys

def _pip(pkg):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', pkg, '--quiet'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'Warning: {pkg} install issue: {r.stderr[-200:]}')

_pip('yfinance')
_pip('tensorflow')
_pip('scikit-learn')

# Also install PyTorch for Feature Selection experiment
torch_cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade',
             'torch', 'torchvision', 'torchaudio',
             '--index-url', 'https://download.pytorch.org/whl/cu124', '--quiet']
r = subprocess.run(torch_cmd, capture_output=True, text=True)
if r.returncode != 0:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade',
                    'torch', '--quiet'], capture_output=True, text=True)

try:
    import pandas_ta
except ModuleNotFoundError:
    _pip('pandas-ta-classic')

print('Packages ready')

In [ ]:
# ============================================================
# Imports & Configuration
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout,
                                      Layer, Permute, Multiply, Flatten,
                                      RepeatVector, Lambda)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ============================================================
# Data Download — TSMC (2330.TW)
# ============================================================
TICKER = '2330.TW'

def download_data(ticker, years=10):
    end = pd.Timestamp.now()
    start = end - pd.Timedelta(days=years * 365 + 14)
    df = yf.download(ticker,
                     start=start.strftime('%Y-%m-%d'),
                     end=(end + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
                     auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index, errors='coerce')
    df = df[~df.index.isna()].copy()
    if getattr(df.index, 'tz', None) is not None:
        df.index = df.index.tz_convert(None)
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=['Open', 'High', 'Low', 'Close'], inplace=True)
    return df

df = download_data(TICKER)
df.to_csv('2330_stock_data.csv')
print(f'Downloaded {len(df)} rows: {df.index[0].date()} ~ {df.index[-1].date()}')
print(df.tail(3))

In [ ]:
# ============================================================
# Train / Test Split
# ============================================================
# Test set: last 10 competition days (2026-03-20 ~ 2026-04-02)
COMP_END = pd.Timestamp('2026-04-02')
TEST_DAYS = 10

close_prices = df['Close'].values.astype(np.float64)

# Find competition end index
comp_end_idx = df.index.get_indexer([COMP_END], method='nearest')[0]
test_start_idx = comp_end_idx - TEST_DAYS + 1

print(f'Total samples: {len(df)}')
print(f'Test range: {df.index[test_start_idx].date()} ~ {df.index[comp_end_idx].date()}')
print(f'Train samples: {test_start_idx}, Test samples: {TEST_DAYS}')

In [ ]:
# ============================================================
# Attention Layer (Keras Implementation)
# ============================================================
class AttentionLayer(Layer):
    """Simple additive attention over LSTM timesteps."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight',
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias',
                                 shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        self.u = self.add_weight(name='att_u',
                                 shape=(input_shape[-1],),
                                 initializer='glorot_uniform', trainable=True)

    def call(self, x):
        score = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        attention_weights = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
        context = tf.reduce_sum(x * tf.expand_dims(attention_weights, -1), axis=1)
        return context

print('AttentionLayer defined')

In [ ]:
# ============================================================
# Model Builder & Training Utilities
# ============================================================

def build_model(look_back, n_features, lstm_units, dropout_rate, lr):
    """Build LSTM + Attention model with given hyperparameters."""
    inp = Input(shape=(look_back, n_features))
    x = inp

    # Stacked LSTM layers (return_sequences=True for all)
    for i, units in enumerate(lstm_units):
        return_seq = True  # all layers return sequences for attention
        x = LSTM(units, return_sequences=return_seq)(x)
        if dropout_rate > 0:
            x = Dropout(dropout_rate)(x)

    # Attention
    x = AttentionLayer()(x)

    # Output
    x = Dense(1)(x)
    model = Model(inp, x)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model


def create_sequences(data, look_back):
    """Create input sequences and targets for LSTM."""
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i + look_back])
        y.append(data[i + look_back, 0])  # predict Close (first column)
    return np.array(X), np.array(y)


def run_experiment(name, look_back, lstm_units, dropout_rate, lr, batch_size,
                   epochs, feature_cols=None, patience=10, verbose=0):
    """Run a complete experiment: prepare data, train, evaluate."""
    if feature_cols is None:
        feature_cols = ['Close']

    # Prepare data
    raw = df[feature_cols].values.astype(np.float64)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(raw[:test_start_idx])          # fit only on train
    scaled = scaler.transform(raw)

    X_all, y_all = create_sequences(scaled, look_back)

    # y_all targets correspond to indices [look_back, look_back+1, ...]
    # Split: train targets have df index < test_start_idx
    target_indices = np.arange(look_back, look_back + len(y_all))

    train_mask = target_indices < test_start_idx
    test_mask = (target_indices >= test_start_idx) & (target_indices <= comp_end_idx)

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    X_test, y_test = X_all[test_mask], y_all[test_mask]

    if len(X_test) == 0:
        print(f'  {name}: No test samples! Skipping.')
        return None

    # Build & Train
    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    model = build_model(look_back, len(feature_cols), lstm_units, dropout_rate, lr)

    es = EarlyStopping(monitor='val_loss', patience=patience,
                       restore_best_weights=True, verbose=0)

    history = model.fit(X_train, y_train,
                        epochs=epochs,
                        batch_size=batch_size,
                        validation_split=0.1,
                        callbacks=[es],
                        verbose=verbose)

    actual_epochs = len(history.history['loss'])

    # Predict & Inverse Transform
    # For inverse: we need to reconstruct the full feature vector
    pred_test_scaled = model.predict(X_test, verbose=0).flatten()
    pred_train_scaled = model.predict(X_train, verbose=0).flatten()

    # Inverse scale (only Close column, index 0)
    def inv_close(scaled_vals, scaler, n_features):
        dummy = np.zeros((len(scaled_vals), n_features))
        dummy[:, 0] = scaled_vals
        return scaler.inverse_transform(dummy)[:, 0]

    n_feat = len(feature_cols)
    pred_test_price = inv_close(pred_test_scaled, scaler, n_feat)
    actual_test_price = inv_close(y_test, scaler, n_feat)
    pred_train_price = inv_close(pred_train_scaled, scaler, n_feat)
    actual_train_price = inv_close(y_train, scaler, n_feat)

    # Metrics
    train_rmse = float(np.sqrt(mean_squared_error(actual_train_price, pred_train_price)))
    train_mape = float(np.mean(np.abs((actual_train_price - pred_train_price) /
                                       (actual_train_price + 1e-8))) * 100)
    test_rmse = float(np.sqrt(mean_squared_error(actual_test_price, pred_test_price)))
    test_mape = float(np.mean(np.abs((actual_test_price - pred_test_price) /
                                      (actual_test_price + 1e-8))) * 100)

    result = {
        'Experiment': name,
        'Epochs': actual_epochs,
        'Train RMSE': round(train_rmse, 4),
        'Train MAPE': f'{train_mape:.2f}%',
        'Test RMSE': round(test_rmse, 4),
        'Test MAPE': f'{test_mape:.2f}%',
    }

    print(f'  {name}: Epochs={actual_epochs}, '
          f'Train RMSE={train_rmse:.4f} MAPE={train_mape:.2f}%, '
          f'Test RMSE={test_rmse:.4f} MAPE={test_mape:.2f}%')

    return result

print('Experiment runner ready')

## 2. Baseline ReproductionDefault settings: Close only, look_back=100, LSTM[128, 64], LR=0.001, Batch=32, Epochs=50, No Dropout.

In [ ]:
# ============================================================
# Baseline Experiment
# ============================================================
print('=' * 80)
print('Baseline: Close only, look_back=100, LSTM[128,64], LR=0.001, Batch=32, Epochs=50')
print('=' * 80)

baseline = run_experiment(
    name='Baseline',
    look_back=100,
    lstm_units=[128, 64],
    dropout_rate=0.0,
    lr=0.001,
    batch_size=32,
    epochs=50,
    patience=10,
)

all_results = [baseline]
print()
print(pd.DataFrame([baseline]).to_string(index=False))

## 3.1 Experiment 1: Sequence Length (look_back)Fix all other parameters at baseline. Test look_back = {60, 90, 120} vs Baseline (100).

In [ ]:
# ============================================================
# Experiment 1: Sequence Length
# ============================================================
print('=' * 80)
print('Experiment 1: Sequence Length (look_back)')
print('=' * 80)

exp1_results = []
for lb in [60, 90, 120]:
    r = run_experiment(
        name=f'look_back={lb}',
        look_back=lb,
        lstm_units=[128, 64],
        dropout_rate=0.0,
        lr=0.001,
        batch_size=32,
        epochs=50,
        patience=10,
    )
    if r:
        exp1_results.append(r)
        all_results.append(r)

print()
df_exp1 = pd.DataFrame([baseline] + exp1_results)
print(df_exp1.to_string(index=False))

## 3.2 Experiment 2: LSTM ArchitectureFix look_back=100. Test different LSTM layer configurations.

In [ ]:
# ============================================================
# Experiment 2: LSTM Architecture
# ============================================================
print('=' * 80)
print('Experiment 2: LSTM Architecture')
print('=' * 80)

arch_configs = [
    ([64, 32], 'LSTM[64,32]'),
    ([256, 128], 'LSTM[256,128]'),
    ([128, 64, 32], 'LSTM[128,64,32]'),
    ([256, 128, 64], 'LSTM[256,128,64]'),
]

exp2_results = []
for units, name in arch_configs:
    r = run_experiment(
        name=name,
        look_back=100,
        lstm_units=units,
        dropout_rate=0.0,
        lr=0.001,
        batch_size=32,
        epochs=50,
        patience=10,
    )
    if r:
        exp2_results.append(r)
        all_results.append(r)

print()
df_exp2 = pd.DataFrame([baseline] + exp2_results)
print(df_exp2.to_string(index=False))

## 3.3 Experiment 3: Training Parameters (LR, Batch Size, Epochs)Fix look_back=100, LSTM[128, 64]. Test variations of learning rate, batch size, and epochs.

In [ ]:
# ============================================================
# Experiment 3: Training Parameters
# ============================================================
print('=' * 80)
print('Experiment 3: Training Parameters (LR, Batch Size, Epochs)')
print('=' * 80)

train_configs = [
    {'name': 'LR=0.0005', 'lr': 0.0005, 'batch_size': 32, 'epochs': 100},
    {'name': 'LR=0.005',  'lr': 0.005,  'batch_size': 32, 'epochs': 50},
    {'name': 'Batch=16',  'lr': 0.001,  'batch_size': 16, 'epochs': 50},
    {'name': 'Batch=64',  'lr': 0.001,  'batch_size': 64, 'epochs': 50},
    {'name': 'Epochs=100','lr': 0.001,  'batch_size': 32, 'epochs': 100},
]

exp3_results = []
for cfg in train_configs:
    r = run_experiment(
        name=cfg['name'],
        look_back=100,
        lstm_units=[128, 64],
        dropout_rate=0.0,
        lr=cfg['lr'],
        batch_size=cfg['batch_size'],
        epochs=cfg['epochs'],
        patience=10,
    )
    if r:
        exp3_results.append(r)
        all_results.append(r)

print()
df_exp3 = pd.DataFrame([baseline] + exp3_results)
print(df_exp3.to_string(index=False))

## 3.4 Experiment 4: Dropout RegularizationFix look_back=100, LSTM[128, 64]. Add Dropout layers after each LSTM.

In [ ]:
# ============================================================
# Experiment 4: Dropout Rate
# ============================================================
print('=' * 80)
print('Experiment 4: Dropout Rate')
print('=' * 80)

exp4_results = []
for dr in [0.1, 0.2, 0.3]:
    r = run_experiment(
        name=f'Dropout={dr}',
        look_back=100,
        lstm_units=[128, 64],
        dropout_rate=dr,
        lr=0.001,
        batch_size=32,
        epochs=60,
        patience=10,
    )
    if r:
        exp4_results.append(r)
        all_results.append(r)

print()
df_exp4 = pd.DataFrame([baseline] + exp4_results)
print(df_exp4.to_string(index=False))

## 3.5 Hyperparameter Tuning SummaryCombined results from all Phase 1 experiments.

In [ ]:
# ============================================================
# Phase 1 Summary Table
# ============================================================
print('=' * 80)
print('Phase 1: Hyperparameter Tuning Summary (All Experiments)')
print('=' * 80)

df_summary = pd.DataFrame(all_results)
print(df_summary.to_string(index=False))

# Find best by Test RMSE
best_idx = df_summary['Test RMSE'].idxmin()
best_row = df_summary.iloc[best_idx]
print(f'\nBest by Test RMSE: {best_row["Experiment"]} (RMSE={best_row["Test RMSE"]}, MAPE={best_row["Test MAPE"]})')

## 4. Feature Selection Experiment (PyTorch Upgrade)Upgrade from Keras to PyTorch. Add 7 technical indicator features with independent normalization.| # | Feature | Normalization | Description ||---|---|---|---|| 1 | Close | log-return z-score | Primary prediction target || 2 | RSI-14 | ÷ 100 | Overbought / Oversold momentum || 3 | MACD | StandardScaler | Mid-term trend || 4 | Volume_ratio | clip(0,5) + MinMax | Volume strength || 5 | Price_range | clip(0,0.2) + MinMax | Intraday volatility || 6 | Open_gap | clip(±10%) + StandardScaler | Overnight gap || 7 | Week_trend | clip(±15%) + StandardScaler | Past week trend |

In [ ]:
# ============================================================
# Feature Selection: PyTorch Multi-Feature LSTM + Attention
# ============================================================
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from copy import deepcopy
import random

try:
    import pandas_ta as ta
except ModuleNotFoundError:
    import pandas_ta_classic as ta

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Device: {DEVICE}')

# ── Technical Indicators ──
df_feat = df.copy()
df_feat.ta.rsi(length=14, append=True)
df_feat.ta.macd(fast=12, slow=26, signal=9, append=True)

macdh_col = next((c for c in df_feat.columns if c.startswith('MACDh')), None)

# Derived features
df_feat['Volume_ratio'] = df_feat['Volume'] / df_feat['Volume'].rolling(20).mean()
df_feat['Price_range'] = (df_feat['High'] - df_feat['Low']) / df_feat['Close']
df_feat['Open_gap'] = (df_feat['Open'] - df_feat['Close'].shift(1)) / df_feat['Close'].shift(1)
df_feat['Week_trend'] = df_feat['Close'].pct_change(5)

# Clean
df_feat.dropna(inplace=True)

FEATURE_COLS_PT = ['Close', 'RSI_14', macdh_col, 'Volume_ratio',
                    'Price_range', 'Open_gap', 'Week_trend']
FEATURE_COLS_PT = [c for c in FEATURE_COLS_PT if c is not None and c in df_feat.columns]
print(f'PyTorch features ({len(FEATURE_COLS_PT)}): {FEATURE_COLS_PT}')

# ── Normalize each feature independently ──
close_feat = df_feat['Close'].values.astype(np.float64)
raw_pt = df_feat[FEATURE_COLS_PT].values.astype(np.float64)

# Recompute test indices for df_feat
comp_end_idx_pt = df_feat.index.get_indexer([COMP_END], method='nearest')[0]
test_start_idx_pt = comp_end_idx_pt - TEST_DAYS + 1

scaler_pt = MinMaxScaler(feature_range=(0, 1))
scaler_pt.fit(raw_pt[:test_start_idx_pt])
scaled_pt = scaler_pt.transform(raw_pt)

print(f'Feature data: {len(df_feat)} rows, test: {df_feat.index[test_start_idx_pt].date()} ~ {df_feat.index[comp_end_idx_pt].date()}')

In [ ]:
# ============================================================
# PyTorch LSTM + Attention Model
# ============================================================
PT_LOOK_BACK = 30
PT_LR = 0.001
PT_EPOCHS = 50
PT_BATCH = 32
PT_SEED = 7

class PTAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.W = nn.Linear(hidden, hidden)
        self.u = nn.Parameter(torch.randn(hidden) * 0.05)

    def forward(self, x):
        score = torch.matmul(torch.tanh(self.W(x)), self.u)
        alpha = torch.softmax(score, dim=1).unsqueeze(-1)
        return (x * alpha).sum(dim=1)


class PTStockLSTM(nn.Module):
    def __init__(self, input_size, lstm_units=[128, 64], dropout_rate=0.2):
        super().__init__()
        self.lstms = nn.ModuleList()
        self.drops = nn.ModuleList()
        in_sz = input_size
        for u in lstm_units:
            self.lstms.append(nn.LSTM(in_sz, u, batch_first=True))
            self.drops.append(nn.Dropout(dropout_rate))
            in_sz = u
        self.attention = PTAttention(lstm_units[-1])
        self.fc = nn.Linear(lstm_units[-1], 1)

    def forward(self, x):
        out = x
        for lstm, drop in zip(self.lstms, self.drops):
            out, _ = lstm(out)
            out = drop(out)
        ctx = self.attention(out)
        return self.fc(ctx).squeeze(-1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


# ── Create dataset (log-return target) ──
def create_dataset_pt(scaled, close, lb):
    X, y, idx = [], [], []
    for i in range(len(scaled) - lb):
        X.append(scaled[i:i+lb])
        log_ret = np.log(close[i+lb] / (close[i+lb-1] + 1e-10))
        y.append(log_ret)
        idx.append(i + lb)
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.float32),
            np.array(idx, dtype=np.int32))


X_pt, y_pt, idx_pt = create_dataset_pt(scaled_pt, close_feat, PT_LOOK_BACK)

tr_mask_pt = idx_pt < test_start_idx_pt
te_mask_pt = (idx_pt >= test_start_idx_pt) & (idx_pt <= comp_end_idx_pt)

X_tr_pt, y_tr_pt = X_pt[tr_mask_pt], y_pt[tr_mask_pt]
X_te_pt, y_te_pt = X_pt[te_mask_pt], y_pt[te_mask_pt]

# Validation split
n_val_pt = max(int(len(X_tr_pt) * 0.1), 20)
X_train_pt = X_tr_pt[:-n_val_pt]
y_train_pt = y_tr_pt[:-n_val_pt]
X_val_pt = X_tr_pt[-n_val_pt:]
y_val_pt = y_tr_pt[-n_val_pt:]

print(f'Train: {X_train_pt.shape}, Val: {X_val_pt.shape}, Test: {X_te_pt.shape}')

# ── Train ──
set_seed(PT_SEED)
model_pt = PTStockLSTM(input_size=len(FEATURE_COLS_PT),
                        lstm_units=[128, 64], dropout_rate=0.2).to(DEVICE)

opt_pt = torch.optim.Adam(model_pt.parameters(), lr=PT_LR)
criterion_pt = nn.HuberLoss(delta=0.01)

X_train_t = torch.FloatTensor(X_train_pt).to(DEVICE)
y_train_t = torch.FloatTensor(y_train_pt).to(DEVICE)
X_val_t = torch.FloatTensor(X_val_pt).to(DEVICE)
y_val_t = torch.FloatTensor(y_val_pt).to(DEVICE)
X_te_t = torch.FloatTensor(X_te_pt).to(DEVICE)

train_loader_pt = DataLoader(TensorDataset(X_train_t, y_train_t),
                              batch_size=PT_BATCH, shuffle=True)

best_val_loss = float('inf')
best_state_pt = None
patience_pt = 10
bad_pt = 0

for ep in range(1, PT_EPOCHS + 1):
    model_pt.train()
    for xb, yb in train_loader_pt:
        opt_pt.zero_grad()
        pred = model_pt(xb)
        loss = criterion_pt(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_pt.parameters(), 1.0)
        opt_pt.step()

    model_pt.eval()
    with torch.no_grad():
        val_pred = model_pt(X_val_t)
        val_loss = criterion_pt(val_pred, y_val_t).item()

    if val_loss < best_val_loss - 1e-8:
        best_val_loss = val_loss
        best_state_pt = deepcopy(model_pt.state_dict())
        bad_pt = 0
    else:
        bad_pt += 1
        if bad_pt >= patience_pt:
            print(f'Early stop at epoch {ep}')
            break

    if ep % 10 == 0:
        print(f'  Epoch {ep}: val_loss={val_loss:.6f}')

if best_state_pt:
    model_pt.load_state_dict(best_state_pt)
print(f'Training complete. Best val loss: {best_val_loss:.6f}')

In [ ]:
# ============================================================
# Feature Selection: Evaluation & Comparison
# ============================================================
model_pt.eval()
with torch.no_grad():
    pred_lr_pt = model_pt(X_te_t).cpu().numpy().flatten()
    pred_lr_tr = model_pt(X_train_t).cpu().numpy().flatten()

# Convert log-return predictions back to prices
idx_te_pt = idx_pt[te_mask_pt]
prev_prices = close_feat[idx_te_pt - 1]
actual_prices = close_feat[idx_te_pt]
pred_prices = prev_prices * np.exp(pred_lr_pt)

# Train metrics (for comparison)
idx_tr_pt_all = idx_pt[tr_mask_pt]
idx_tr_pt_train = idx_tr_pt_all[:-n_val_pt]
prev_tr = close_feat[idx_tr_pt_train - 1]
actual_tr = close_feat[idx_tr_pt_train]
pred_tr = prev_tr * np.exp(pred_lr_tr)

train_rmse_pt = float(np.sqrt(mean_squared_error(actual_tr, pred_tr)))
train_mape_pt = float(np.mean(np.abs((actual_tr - pred_tr) / (actual_tr + 1e-8))) * 100)
test_rmse_pt = float(np.sqrt(mean_squared_error(actual_prices, pred_prices)))
test_mape_pt = float(np.mean(np.abs((actual_prices - pred_prices) / (actual_prices + 1e-8))) * 100)

print('=' * 80)
print('Feature Selection: Comparison Results')
print('=' * 80)

comparison = pd.DataFrame([
    {'Model': 'Close only (Baseline, Keras)',
     'Train RMSE': baseline['Train RMSE'] if baseline else 'N/A',
     'Train MAPE': baseline['Train MAPE'] if baseline else 'N/A',
     'Test RMSE': baseline['Test RMSE'] if baseline else 'N/A',
     'Test MAPE': baseline['Test MAPE'] if baseline else 'N/A'},
    {'Model': f'{len(FEATURE_COLS_PT)}-Feature (PyTorch)',
     'Train RMSE': round(train_rmse_pt, 4),
     'Train MAPE': f'{train_mape_pt:.2f}%',
     'Test RMSE': round(test_rmse_pt, 4),
     'Test MAPE': f'{test_mape_pt:.2f}%'},
])
print(comparison.to_string(index=False))

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
days = range(1, len(actual_prices) + 1)
ax.plot(days, actual_prices, 'b-o', label='Actual', markersize=5)
ax.plot(days, pred_prices, 'r--s', label=f'Predicted ({len(FEATURE_COLS_PT)} features)', markersize=5)
ax.set_xlabel('Day')
ax.set_ylabel('Close Price (TWD)')
ax.set_title('Feature Selection: Multi-Feature PyTorch Model vs Actual')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nPhase 1 experiments complete.')